In [5]:
# Fine tune modernbert base embed model with squad data to create a fine tuned embedding model
# which we will compare with the default sentence transformer model

import pandas as pd
# CSVs with the format question,context, answers
train_df_all = pd.read_csv("../data/squad_v2/train.csv")
val_df_all = pd.read_csv("../data/squad_v2/validation.csv")

for df in (train_df_all, val_df_all):
    df["question"] = "search_query: " + df["question"]
    df["context"] = "search_document: " + df["context"]


len(train_df_all), len(val_df_all)

(130319, 11873)

In [6]:
# Filter those rows which are answerable from squad v2 dataset, so as to take only those query and context pair for proper training
answer_col = "answers" 
pattern = r"""['"]text['"]\s*:\s*array\(\s*\[\s*['"]"""
train_df = train_df_all[
    train_df_all[answer_col].str.contains(pattern, regex=True, na=False)
].copy()
val_df = val_df_all[
    val_df_all[answer_col].str.contains(pattern, regex=True, na=False)
].copy()
len(train_df), len(val_df)

(86821, 5928)

In [7]:
# Sample N records
train_df.head(1000)

,id,title,context,question,answers
0,56be85543aeaaa14008c9063,Beyoncé,search_document: Beyoncé Giselle Knowles-Carte...,search_query: When did Beyonce start becoming ...,"{'text': array(['in the late 1990s'], dtype=ob..."
1,56be85543aeaaa14008c9065,Beyoncé,search_document: Beyoncé Giselle Knowles-Carte...,search_query: What areas did Beyonce compete i...,"{'text': array(['singing and dancing'], dtype=..."
2,56be85543aeaaa14008c9066,Beyoncé,search_document: Beyoncé Giselle Knowles-Carte...,search_query: When did Beyonce leave Destiny's...,"{'text': array(['2003'], dtype=object), 'answe..."
3,56bf6b0f3aeaaa14008c9601,Beyoncé,search_document: Beyoncé Giselle Knowles-Carte...,search_query: In what city and state did Beyon...,"{'text': array(['Houston, Texas'], dtype=objec..."
4,56bf6b0f3aeaaa14008c9602,Beyoncé,search_document: Beyoncé Giselle Knowles-Carte...,search_query: In which decade did Beyonce beco...,"{'text': array(['late 1990s'], dtype=object), ..."
...,...,...,...,...,...
995,56d319d459d6e41400146241,Frédéric_Chopin,"search_document: In Paris, Chopin encountered ...",search_query: What position did the poet who C...,{'text': array(['principal of the Polish Liter...
996,56cbf5106d243a140015ee18,Frédéric_Chopin,search_document: Two Polish friends in Paris w...,search_query: Which friend of Frédéric failed ...,"{'text': array(['Julian Fontana'], dtype=objec..."
997,56cbf5106d243a140015ee19,Frédéric_Chopin,search_document: Two Polish friends in Paris w...,search_query: Who was Frédéric's trusted advis...,"{'text': array(['Albert Grzymała'], dtype=obje..."
998,56cbf5106d243a140015ee1a,Frédéric_Chopin,search_document: Two Polish friends in Paris w...,search_query: What familial role was Albert Gr...,"{'text': array(['elder brother'], dtype=object..."


In [15]:
# Taking 50 samples each in train and validation
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df[["question", "context"]].head(75).rename(columns={'question': 'query'}), preserve_index=False)
val_dataset =  Dataset.from_pandas(val_df[["question", "context"]].head(50).rename(columns={'question': 'query'}), preserve_index=False)
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(val_dataset)}")
train_dataset, val_dataset


Train dataset size: 75
Test dataset size: 50


(Dataset({
     features: ['query', 'context'],
     num_rows: 75
 }),
 Dataset({
     features: ['query', 'context'],
     num_rows: 50
 }))

In [16]:
# Fine Tuning 
import torch
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers import SentenceTransformer

# locally saved pretrained model
model_name = "../model/modernbert-embed-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
# For query & context pair, multiple negative ranking loss is desired
loss = MultipleNegativesRankingLoss(model)


Loading weights: 100%|██████████| 134/134 [00:03<00:00, 35.35it/s]


In [17]:
from sentence_transformers import SentenceTransformerTrainingArguments , SentenceTransformerTrainer

args = SentenceTransformerTrainingArguments(
    output_dir="../output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=False,
    eval_strategy="steps",
    eval_steps=0.125,
    save_strategy="steps",
    save_steps=0,
    save_total_limit=2,
    logging_steps=1,
    report_to="none"
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=loss,
)



trainer.train()

H:\pythonProject\natural_language_processing\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
15,0.003241,0.198016
30,0.000783,0.219972
45,0.195937,0.227095
60,0.000039,0.220143
75,0.001129,0.218207
90,0.000827,0.218635
105,0.001485,0.217582
114,0.000000,0.217751


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.30s/it]


TrainOutput(global_step=114, training_loss=0.17617061054497435, metrics={'train_runtime': 1931.5852, 'train_samples_per_second': 0.116, 'train_steps_per_second': 0.059, 'total_flos': 0.0, 'train_loss': 0.17617061054497435, 'epoch': 3.0})

In [18]:
# Prediction section comparing various models for similarity

search_query_str = "search_query: "
search_document_str= "search_document: "
query_list = ["what is the maturity date for abc inc?"]
document_list = ["The date at which india got independence is 15th august 1947", 
                 "The maturity date is February 26th 2019", 
                 "Abc inc's maturity date is 2nd June 2021"]

query_list_final = [search_query_str + x for x in query_list]
document_list_final = [search_document_str + x for x in document_list]



# Evaluate which is the best model, with fine tuning the domain specific fine tuned model should be better
fine_tuned_model = "../output/checkpoint-114"
base_model = "../model/modernbert-embed-base"
generic_embedding_model = "../model/all-MiniLM-L6-v2"
all_models = [fine_tuned_model, base_model, generic_embedding_model]

for m in all_models:
    print("Model:", m)
    model = SentenceTransformer(m, device=device)
    
    if "all-MiniLM-L6-v2" not in m:
        query_embed = model.encode(query_list_final)
        doc_embed = model.encode(document_list_final)
    else:
        query_embed = model.encode(query_list)
        doc_embed = model.encode(document_list)
        
    print(query_embed.shape, doc_embed.shape)
    similarity = model.similarity(query_embed, doc_embed)
    print("Similarity", similarity)

Model: ../output/checkpoint-114


Loading weights: 100%|██████████| 134/134 [00:01<00:00, 88.03it/s]


(1, 768) (3, 768)
Similarity tensor([[0.2776, 0.6184, 0.8566]])
Model: ../model/modernbert-embed-base


Loading weights: 100%|██████████| 134/134 [00:01<00:00, 106.31it/s]


(1, 768) (3, 768)
Similarity tensor([[0.2363, 0.5766, 0.8409]])
Model: ../model/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2228.72it/s]


(1, 384) (3, 384)
Similarity tensor([[0.1430, 0.6805, 0.8989]])
